# Task 04 — Sentiment Analysis and Visualization
Analyze sentiment patterns in social media data using the official Prodigy InfoTech Task 04 dataset.

In [1]:
import re
import pandas as pd
import matplotlib.pyplot as plt
DATA_URL = "https://raw.githubusercontent.com/Prodigy-InfoTech/data-science-datasets/main/Task%204/twitter_training.csv"
df = pd.read_csv(DATA_URL, header=None, names=["ID", "Topic", "Sentiment", "Tweet"])
print("Original shape:", df.shape)
print(df.isna().sum())

Original shape: (74682, 4)
ID            0
Topic         0
Sentiment      0
Tweet       686
dtype: int64


## Cleaning and preprocessing

In [1]:
df = df.dropna(subset=["Topic", "Sentiment", "Tweet"]).copy()
df["Topic"] = df["Topic"].astype(str).str.strip()
df["Sentiment"] = df["Sentiment"].astype(str).str.strip().str.title()
df["Tweet"] = df["Tweet"].astype(str).str.strip()
df = df[df["Tweet"].ne("")]
df = df[df["Sentiment"].isin(["Positive", "Negative", "Neutral", "Irrelevant"])].copy()
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#\w+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()
df["Clean_Tweet"] = df["Tweet"].map(clean_text)
df["Tweet_Length"] = df["Clean_Tweet"].str.len()
print("Rows after cleaning:", len(df))
print("Unique topics:", df["Topic"].nunique())

Rows after cleaning: 73824
Unique topics: 32


## Genuine results

In [1]:
counts = df["Sentiment"].value_counts()
print(counts)
counts.plot(kind="bar", figsize=(8,5), title="Overall Social Media Sentiment Distribution")
plt.xlabel("Sentiment"); plt.ylabel("Number of posts"); plt.tight_layout(); plt.show()

Negative      22312
Positive      20619
Neutral       18051
Irrelevant    12842
Name: Sentiment, dtype: int64


## Sentiment by topic/entity

In [1]:
top_topics = df["Topic"].value_counts().head(10).index
table = pd.crosstab(df.loc[df["Topic"].isin(top_topics), "Topic"], df.loc[df["Topic"].isin(top_topics), "Sentiment"])
table = table.reindex(columns=["Positive","Negative","Neutral","Irrelevant"], fill_value=0)
print(table)
table.plot(kind="bar", figsize=(12,6)); plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

                         Positive  Negative  Neutral  Irrelevant
Topic
ApexLegends                   632       589      934         192
CallOfDuty                    445       880      374         672
Dota2                         579       767      588         425
Facebook                      174       715      786         685
LeagueOfLegends               614       629      817         312
MaddenNFL                     396      1689      195          90
Microsoft                     601       761      818         169
TomClancysRainbowSix          509      1113      636          96
Verizon                       528      1091      564         178
WorldOfCraft                  737       340     1063         216


## Tweet length by sentiment

In [1]:
plt.figure(figsize=(9,6))
for sentiment in ["Positive","Negative","Neutral","Irrelevant"]:
    values = df.loc[df["Sentiment"] == sentiment, "Tweet_Length"]
    if not values.empty: plt.hist(values, bins=30, alpha=.45, label=sentiment)
plt.xlabel("Cleaned tweet length (characters)"); plt.ylabel("Number of posts"); plt.legend(); plt.tight_layout(); plt.show()